In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os

# -----------------------------------------------------------
def resize_nearest(img, new_h, new_w):
    old_h, old_w = img.shape[:2]
    row_idx = (np.linspace(0, old_h - 1, new_h)).astype(int)
    col_idx = (np.linspace(0, old_w - 1, new_w)).astype(int)
    grid_y, grid_x = np.meshgrid(row_idx, col_idx, indexing='ij')
    return img[grid_y, grid_x]

def compute_Roundness(mask):
    area = np.sum(mask)
    padded = np.pad(mask, pad_width=1, mode='constant', constant_values=0)
    perimeter_mask = (
        mask &
        (
            (~padded[:-2, 1:-1]) |
            (~padded[2:,  1:-1]) |
            (~padded[1:-1, :-2]) |
            (~padded[1:-1, 2:])
        )
    )
    perimeter = np.sum(perimeter_mask)
    return (4 * np.pi * area / (perimeter ** 2)) if perimeter > 0 else 0

def compute_Elongation(mask):
    ys, xs  = np.where(mask)
    coords  = np.column_stack((xs, ys))
    cov     = np.cov(coords, rowvar=False)
    eigvals = np.linalg.eigvalsh(cov)
    return np.sqrt(eigvals[1]) / np.sqrt(eigvals[0]) if eigvals[0] > 0 else np.inf

def FeatureExtraction(img_raw):
    if img_raw.dtype != np.uint8:
        img_raw = (img_raw * 255).astype(np.uint8)
    if img_raw.shape[2] == 4:
        img_raw = img_raw[:, :, :3]
    img  = resize_nearest(img_raw, 200, 200)
    gray = np.dot(img[..., :3], [0.2989, 0.5870, 0.1140]).astype(np.uint8)
    hist, _ = np.histogram(gray.ravel(), bins=256, range=(0, 256))
    total = gray.size
    current_max, threshold = 0, 0
    sum_total = np.dot(np.arange(256), hist)
    sum_bg, w_bg = 0.0, 0.0
    for i in range(256):
        w_bg += hist[i]
        if w_bg == 0: continue
        w_fg = total - w_bg
        if w_fg == 0: break
        sum_bg += i * hist[i]
        m_bg = sum_bg / w_bg
        m_fg = (sum_total - sum_bg) / w_fg
        bv   = w_bg * w_fg * (m_bg - m_fg) ** 2
        if bv > current_max:
            current_max = bv
            threshold   = i
    mask          = gray < threshold
    roundness     = compute_Roundness(mask)
    elongation    = compute_Elongation(mask)
    object_pixels = img[mask]
    if object_pixels.size > 0:
        avg_r, avg_g, avg_b = object_pixels.mean(axis=0)
    else:
        avg_r, avg_g, avg_b = -1, -1, -1
    return np.array([roundness, elongation, avg_r, avg_g, avg_b])

def normal_pdf(x, mu, sigma):
    return (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-((x - mu) ** 2) / (2 * sigma ** 2))

class NaiveBayesClassifier:
    def __init__(self, _DataLoc, _ClassName):
        self.DataLoc   = _DataLoc
        self.ClassName = _ClassName

    def compute_posterior_probability(self, queried_x):
        df  = pd.read_csv(self.DataLoc)
        X   = df.iloc[:, :-1].to_numpy(dtype=float)
        y   = df.iloc[:, -1].to_numpy(str)
        N   = y.size
        class_labels    = np.unique(y)
        posterior_probs = []
        for c in class_labels:
            P_c        = np.sum(y == c) / N
            indices    = np.where(y == c)[0]
            likelihood = 1.0
            for i in range(len(queried_x)):
                mu    = np.mean(X[indices, i])
                sigma = np.std(X[indices, i])
                sigma = sigma if sigma > 0 else 1e-6
                likelihood *= normal_pdf(queried_x[i], mu, sigma)
            posterior_probs.append(P_c * likelihood)
        posterior_probs = np.array(posterior_probs)
        total = np.sum(posterior_probs)
        posterior_probs = posterior_probs / total if total > 0 else np.zeros_like(posterior_probs)
        return posterior_probs

# ======================================================
CLASS_NAMES  = ['Canada', 'Japan', 'Korea', 'USA', 'Vietnam']
CLASS_COLORS = ['red', 'orange', 'green', 'blue', 'purple']
CSV_PATH     = 'currency_feature_dataset.csv'

classifier  = NaiveBayesClassifier(CSV_PATH, CLASS_NAMES)
total_ok    = 0
total_wrong = 0

for class_name in CLASS_NAMES:
    folder_path = 'images/' + class_name
    file_list   = sorted([f for f in os.listdir(folder_path) if f.lower().endswith('.bmp')])
    class_ok    = 0
    class_wrong = 0

    print(f'\n===== {class_name} ({len(file_list)} anh) =====')

    for file_name in file_list:
        img_path  = folder_path + '/' + file_name
        img_raw   = plt.imread(img_path)
        query     = FeatureExtraction(img_raw)
        probs     = classifier.compute_posterior_probability(query)
        predicted = CLASS_NAMES[np.argmax(probs)]

        if predicted == class_name:
            status = '[OK   ]'
            class_ok    += 1
            total_ok    += 1
        else:
            status = '[WRONG]'
            class_wrong += 1
            total_wrong += 1

        print(f'  {status} {file_name:15s} -> {predicted}')

        # Hien thi anh + bieu do
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        fig.suptitle(f'[{class_name}] {file_name}  ->  Predicted: {predicted} {status}', fontsize=12)
        axes[0].imshow(img_raw)
        axes[0].axis('off')
        axes[0].set_title('Input Image')
        axes[1].bar(CLASS_NAMES, probs, color=CLASS_COLORS)
        axes[1].set_ylim(0, 1.2)
        axes[1].set_xlabel('Class')
        axes[1].set_ylabel('Posterior Probability')
        axes[1].set_title('Class Probabilities')
        for i, v in enumerate(probs):
            axes[1].text(i, v + 0.02, f'{v:.2f}', ha='center', va='bottom', fontsize=11)
        plt.tight_layout()
        plt.show()

    print(f'  >> {class_name}: OK={class_ok}, WRONG={class_wrong}')

# ======================================================
total = total_ok + total_wrong
print('\n==============================')
print(f'TONG KET:')
print(f'  Tong anh  : {total}')
print(f'  OK        : {total_ok}  ({total_ok/total*100:.1f}%)')
print(f'  WRONG     : {total_wrong}  ({total_wrong/total*100:.1f}%)')
print(f'  Accuracy  : {total_ok/total*100:.2f}%')
print('==============================')
